In [ ]:
import matplotlib.pyplot as plt
import random
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.metrics import log_loss
from sklearn.ensemble import RandomForestClassifier

import sys

sys.path.append("./..")

from src.generate_data import generate_dataset

rng = random.Random(213)

In [3]:
X, y, _ = generate_dataset()

[6, 2, 1, 3, 8]

informative [0, 1, 2, 3, 4, 5]
redundant [6, 7]
correlated [8]
noise [9, 10, 11]
pure_noise [12, 13, 14, 15, 16, 17, 18, 19]


In [ ]:
seed = rng.randint(1, 10000)
X, y, feature_types = generate_dataset(n_samples=500, n_features=20, random_state=seed)
num_of_informative_features = len(feature_types["informative"])

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed
)

# Store results
selected_features = {}

# --- 1. LASSO ---
lasso = LassoCV(cv=5, random_state=seed, max_iter=10000).fit(X_train, y_train)
lasso_selected = np.where(lasso.coef_ != 0)[0].tolist()
selected_features["lasso"] = lasso_selected

# --- 2. Mutual Information ---
mi_scores = mutual_info_classif(X_train, y_train, random_state=seed)
mi_selected = np.argsort(mi_scores)[-num_of_informative_features:].tolist()
selected_features["mutual_info"] = mi_selected

# --- 3. Correlation ---
correlations = []
for i in range(X_train.shape[1]):
    corr = np.corrcoef(X_train[:, i], y_train)[0, 1]
    correlations.append((i, abs(corr)))

correlated = [
    i
    for i, _ in sorted(correlations, key=lambda x: -x[1])[:num_of_informative_features]
]
selected_features["correlation"] = correlated


# --- 4. BIC Wrapper (Logistic Regression with feature subsets) ---
def bic_score(model, X, y):
    """Calculate BIC for a fitted logistic regression model"""
    n = len(y)
    k = X.shape[1]
    prob = model.predict_proba(X)[:, 1]
    ll = -log_loss(y, prob, normalize=False)  # log-likelihood
    bic = k * np.log(n) - 2 * ll
    return bic


def bic_forward_selection(X, y, max_features=None):
    selected = []
    remaining = list(range(X.shape[1]))
    best_bic = np.inf

    while remaining:
        scores = []
        for feat in remaining:
            candidate = selected + [feat]
            model = LogisticRegression(solver="liblinear", max_iter=10000).fit(
                X[:, candidate], y
            )
            bic = bic_score(model, X[:, candidate], y)
            scores.append((bic, feat))

        # Find the best new feature to add
        scores.sort()
        best_new_bic, best_feat = scores[0]

        # Stop if BIC does not improve
        if best_new_bic < best_bic:
            best_bic = best_new_bic
            selected.append(best_feat)
            remaining.remove(best_feat)
            if max_features and len(selected) >= max_features:
                break
        else:
            break

    return selected


bic_selected = bic_forward_selection(X_train, y_train)
selected_features["bic"] = bic_selected

# --- 5. Random Forest Feature Importance ---
rf = RandomForestClassifier(n_estimators=100, random_state=seed)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
rf_selected = np.argsort(importances)[-num_of_informative_features:].tolist()
selected_features["random_forest"] = rf_selected

# --- Print results ---
print()
for method, features in selected_features.items():
    print(f"{method}: selected feature indices: {features}")